# EduPredict-XAI

## Step 19 — Final Prediction Pipeline

### Main Model
TabPFN

### XAI Technique
SHAP

### Target
final_exam_score

### Objective
Create a reusable prediction pipeline for the final application.

In [2]:
# Import Libraries

import os
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 60)
print("STEP 19 — FINAL PREDICTION PIPELINE")
print("=" * 60)

print("\nPython version:")
print(sys.version.split()[0])

print("\nLibraries imported successfully! ")

STEP 19 — FINAL PREDICTION PIPELINE

Python version:
3.10.9

Libraries imported successfully! 


In [3]:
# Find Project Root

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("=" * 60)
print("PROJECT ROOT")
print("=" * 60)

print(PROJECT_ROOT)

if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError(
        "Project root could not be identified."
    )

print("\nProject root verified successfully!")

PROJECT ROOT
d:\MCA Sem 3\RP\EduPredict-XAI

Project root verified successfully!


In [4]:
#  Define Project Paths

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "student_performance_feature_engineered.csv"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
)

RESULTS_PATH = (
    PROJECT_ROOT
    / "results"
)

XAI_RESULTS_PATH = (
    RESULTS_PATH
    / "xai"
)

print("=" * 60)
print("PROJECT PATHS")
print("=" * 60)

print("\nData:")
print(DATA_PATH)

print("\nModels:")
print(MODEL_PATH)

print("\nResults:")
print(RESULTS_PATH)

print("\nXAI:")
print(XAI_RESULTS_PATH)

PROJECT PATHS

Data:
d:\MCA Sem 3\RP\EduPredict-XAI\data\processed\student_performance_feature_engineered.csv

Models:
d:\MCA Sem 3\RP\EduPredict-XAI\models

Results:
d:\MCA Sem 3\RP\EduPredict-XAI\results

XAI:
d:\MCA Sem 3\RP\EduPredict-XAI\results\xai


In [5]:
# Load Dataset

df = pd.read_csv(
    DATA_PATH
)

TARGET = "final_exam_score"

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("\nShape:")
print(df.shape)

print("\nTarget:")
print(TARGET)

print("\nColumns:")
print(df.columns.tolist())

DATASET LOADED

Shape:
(1000, 10)

Target:
final_exam_score

Columns:
['gender', 'study_time_hours', 'attendance_percent', 'sleep_hours', 'parental_education', 'internet_access', 'extracurricular_activities', 'part_time_job', 'previous_grade', 'final_exam_score']


In [6]:
# Define Final Feature List

X = df.drop(
    columns=[TARGET]
).copy()

y = df[TARGET].copy()

FEATURES = X.columns.tolist()

print("=" * 60)
print("FINAL MODEL FEATURES")
print("=" * 60)

for i, feature in enumerate(
    FEATURES,
    start=1
):
    print(f"{i}. {feature}")

print("\nTotal features:", len(FEATURES))

FINAL MODEL FEATURES
1. gender
2. study_time_hours
3. attendance_percent
4. sleep_hours
5. parental_education
6. internet_access
7. extracurricular_activities
8. part_time_job
9. previous_grade

Total features: 9


In [7]:
# Load Final TabPFN Model

TABPFN_MODEL_PATH = (
    MODEL_PATH
    / "tabpfn_fitted_model.pkl"
)

print("=" * 60)
print("LOADING FINAL TABPFN MODEL")
print("=" * 60)

print("\nPath:")
print(TABPFN_MODEL_PATH)

if not TABPFN_MODEL_PATH.exists():
    raise FileNotFoundError(
        "tabpfn_fitted_model.pkl was not found."
    )

with open(
    TABPFN_MODEL_PATH,
    "rb"
) as file:

    tabpfn_model = pickle.load(file)

print("\nTabPFN model loaded successfully!")
print("\nModel type:")
print(type(tabpfn_model))

LOADING FINAL TABPFN MODEL

Path:
d:\MCA Sem 3\RP\EduPredict-XAI\models\tabpfn_fitted_model.pkl


d:\MCA Sem 3\RP\EduPredict-XAI\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



TabPFN model loaded successfully!

Model type:
<class 'tabpfn.regressor.TabPFNRegressor'>


In [8]:
# Verify TabPFN Model

print("=" * 60)
print("VERIFYING TABPFN")
print("=" * 60)

test_data = X.iloc[:3].copy()

test_predictions = tabpfn_model.predict(
    test_data
)

print("\nInput shape:")
print(test_data.shape)

print("\nPredictions:")
print(test_predictions)

if len(test_predictions) != 3:
    raise ValueError(
        "TabPFN verification failed."
    )

print("\nTabPFN is ready for prediction!")

VERIFYING TABPFN


TabPFN inference: 100%|██████████| 8/8 [00:11<00:00,  1.44s/estimator]


Input shape:
(3, 9)

Predictions:
[94.81946 98.64557 96.0624 ]

TabPFN is ready for prediction!


In [9]:
# Input Validation Function

def validate_student_input(
    student_input
):

    if not isinstance(
        student_input,
        pd.DataFrame
    ):
        raise TypeError(
            "Student input must be a pandas DataFrame."
        )

    missing_features = [
        feature
        for feature in FEATURES
        if feature not in student_input.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing features: {missing_features}"
        )

    extra_features = [
        column
        for column in student_input.columns
        if column not in FEATURES
    ]

    if extra_features:
        print(
            "Warning: Extra features will be ignored:",
            extra_features
        )

    student_input = student_input[
        FEATURES
    ].copy()

    if student_input.isnull().any().any():

        missing = student_input.columns[
            student_input.isnull().any()
        ].tolist()

        raise ValueError(
            f"Missing values found in: {missing}"
        )

    return student_input


print(
    "Input validation function created successfully!"
)

Input validation function created successfully!


In [10]:
# Final Prediction Function

def predict_student_performance(
    student_input
):

    student_input = validate_student_input(
        student_input
    )

    prediction = tabpfn_model.predict(
        student_input
    )

    return float(
        prediction[0]
    )


print(
    "Prediction function created successfully!"
)

Prediction function created successfully!


In [11]:
# Create Test Student

test_student = X.iloc[
    [0]
].copy()

print("=" * 60)
print("TEST STUDENT")
print("=" * 60)

display(
    test_student
)

TEST STUDENT


,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade
0,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9


In [12]:
# Test Prediction

predicted_score = (
    predict_student_performance(
        test_student
    )
)

actual_score = float(
    y.iloc[0]
)

print("=" * 60)
print("STUDENT PERFORMANCE PREDICTION")
print("=" * 60)

print("\nActual final exam score:")
print(round(actual_score, 2))

print("\nPredicted final exam score:")
print(round(predicted_score, 2))

print("\nPrediction completed successfully!")

TabPFN inference: 100%|██████████| 8/8 [00:13<00:00,  1.63s/estimator]

STUDENT PERFORMANCE PREDICTION

Actual final exam score:
100.0

Predicted final exam score:
94.82

Prediction completed successfully!


In [13]:
# Performance Category

def performance_category(
    score
):

    if score >= 75:
        return "Excellent"

    elif score >= 60:
        return "Good"

    elif score >= 50:
        return "Average"

    elif score >= 35:
        return "Needs Improvement"

    else:
        return "At Risk"


category = performance_category(
    predicted_score
)

print("=" * 60)
print("PERFORMANCE CATEGORY")
print("=" * 60)

print("\nPredicted score:")
print(round(predicted_score, 2))

print("\nPerformance category:")
print(category)

PERFORMANCE CATEGORY

Predicted score:
94.82

Performance category:
Excellent


In [14]:
# Create Prediction Result

prediction_result = pd.DataFrame({
    "Predicted_Final_Exam_Score": [
        predicted_score
    ],
    "Performance_Category": [
        category
    ]
})

print("=" * 60)
print("PREDICTION RESULT")
print("=" * 60)

display(
    prediction_result
)

PREDICTION RESULT


,Predicted_Final_Exam_Score,Performance_Category
0,94.819443,Excellent


In [15]:
# Save Example Prediction

FINAL_PIPELINE_RESULTS = (
    RESULTS_PATH
    / "final_pipeline"
)

FINAL_PIPELINE_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)

example_prediction_path = (
    FINAL_PIPELINE_RESULTS
    / "example_student_prediction.csv"
)

prediction_result.to_csv(
    example_prediction_path,
    index=False
)

print(
    "Example prediction saved successfully!"
)

print(
    example_prediction_path
)

Example prediction saved successfully!
d:\MCA Sem 3\RP\EduPredict-XAI\results\final_pipeline\example_student_prediction.csv


In [16]:
# Model Metadata

model_metadata = {
    "model": "TabPFN",
    "task": "Regression",
    "target": TARGET,
    "number_of_features": len(FEATURES),
    "features": FEATURES,
    "test_size": 0.20,
    "random_state": 42,
    "primary_xai": "SHAP"
}

metadata_df = pd.DataFrame({
    "Property": model_metadata.keys(),
    "Value": [
        str(value)
        for value in model_metadata.values()
    ]
})

print("=" * 60)
print("FINAL MODEL METADATA")
print("=" * 60)

display(
    metadata_df
)

FINAL MODEL METADATA


,Property,Value
0,model,TabPFN
1,task,Regression
2,target,final_exam_score
3,number_of_features,9
4,features,"['gender', 'study_time_hours', 'attendance_per..."
5,test_size,0.2
6,random_state,42
7,primary_xai,SHAP


In [17]:
# Save Model Metadata

metadata_path = (
    FINAL_PIPELINE_RESULTS
    / "model_metadata.csv"
)

metadata_df.to_csv(
    metadata_path,
    index=False
)

print(
    "Model metadata saved successfully!"
)

print(metadata_path)

Model metadata saved successfully!
d:\MCA Sem 3\RP\EduPredict-XAI\results\final_pipeline\model_metadata.csv


In [18]:
# Complete Pipeline Test

print("=" * 70)
print("COMPLETE PREDICTION PIPELINE TEST")
print("=" * 70)

# 1. Input
student = X.iloc[
    [0]
].copy()

# 2. Validation
student = validate_student_input(
    student
)

# 3. Prediction
score = predict_student_performance(
    student
)

# 4. Category
category = performance_category(
    score
)

print("\nINPUT:")
display(student)

print("\nPREDICTED SCORE:")
print(round(score, 2))

print("\nPERFORMANCE CATEGORY:")
print(category)

print("\n" + "=" * 70)
print("COMPLETE PIPELINE TEST SUCCESSFUL!")
print("=" * 70)

COMPLETE PREDICTION PIPELINE TEST


TabPFN inference: 100%|██████████| 8/8 [00:12<00:00,  1.54s/estimator]


INPUT:


,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade
0,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9



PREDICTED SCORE:
94.82

PERFORMANCE CATEGORY:
Excellent

COMPLETE PIPELINE TEST SUCCESSFUL!


In [19]:
# Step 19 Completion Checkpoint

print("=" * 70)
print("STEP 19 — FINAL PREDICTION PIPELINE COMPLETED")
print("=" * 70)

print("\nMain Model:")
print("TabPFN")

print("\nTarget:")
print(TARGET)

print("\nNumber of Features:")
print(len(FEATURES))

print("\nXAI:")
print("SHAP")

print("\nPipeline:")
print(
    "Input → Validation → TabPFN → "
    "Predicted Score → Performance Category"
)

print("\nFiles created:")
print("✓ example_student_prediction.csv")
print("✓ model_metadata.csv")

print("\n" + "=" * 70)
print("STEP 19 COMPLETED SUCCESSFULLY!")
print("=" * 70)

STEP 19 — FINAL PREDICTION PIPELINE COMPLETED

Main Model:
TabPFN

Target:
final_exam_score

Number of Features:
9

XAI:
SHAP

Pipeline:
Input → Validation → TabPFN → Predicted Score → Performance Category

Files created:
✓ example_student_prediction.csv
✓ model_metadata.csv

STEP 19 COMPLETED SUCCESSFULLY!
